In [ ]:

import numpy as np
import pandas as pd
import os


## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Importing Dataset

In [ ]:
dataset_train = pd.read_csv("/content/test_Y3wMUE5_7gLdaTN.csv")
dataset_test = pd.read_csv("/content/train_u6lujuX_CVtuZ9i.csv")

## EDA

In [ ]:
dataset_train.head()

In [ ]:
dataset_train.info()

In [ ]:
dataset_train.describe()

In [ ]:
dataset_train.shape

In [ ]:
dataset_train.isnull().sum()

In [ ]:
dataset = dataset_train.dropna()

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.replace({"Loan_Status":{'N':0,'Y':1}},inplace=True)

In [ ]:
dataset.head()

In [ ]:
# define numerical & categorical columns
numeric_features = [feature for feature in dataset.columns if dataset[feature].dtype != 'O']
categorical_features = [feature for feature in dataset.columns if dataset[feature].dtype == 'O']

# print columns
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

In [ ]:
for i in categorical_features:
    if i == 'Loan_ID':
        continue
    print(i,dataset[i].unique())

In [ ]:
for i in numeric_features:
    if i == 'Loan_ID':
        continue
    print(i,dataset[i].unique())

In [ ]:
dataset['Dependents'].value_counts()

In [ ]:
dataset = dataset.replace(to_replace='3+', value=4)

In [ ]:
# convert categorical columns to numerical values
dataset.replace({'Married':{'No':0,'Yes':1},'Gender':{'Male':1,'Female':0},'Self_Employed':{'No':0,'Yes':1},
                      'Property_Area':{'Rural':0,'Semiurban':1,'Urban':2},'Education':{'Graduate':1,'Not Graduate':0}},inplace=True)

In [ ]:
dataset.head()

In [ ]:
dataset.dtypes


In [ ]:
dataset.head()

In [ ]:
dataset.describe()

In [ ]:
# dataset['LoanAmount'].median()
dataset['LoanAmount'].mode()

## Data Visualization

In [ ]:
sns.displot(dataset['LoanAmount'],kde=True)

In [ ]:
sns.displot(dataset['ApplicantIncome'],kde=True)

**Insight**
* Average applicant Income is 5364.231, min is 150 and max is 81000.
* Maximum loanAmount  person have is 600, minimum is 9 and mean is 144.
* Median loanAmount is 128, Mode is 100,110,120.
* We can say maximum number of ammount is between 100-200.
* Large number of applicant have income between 0-10000.
* maximum applicant income is 81000 and minimum is 150.

In [ ]:
# education & Loan Status
sns.countplot(x='Education',hue='Loan_Status',data=dataset)

In [ ]:
# marital status & Loan Status

sns.countplot(x='Married',hue='Loan_Status',data=dataset)

In [ ]:
# Gender = 1=Male,0=Female
# Marriage 0=NO,1=Yes

sns.countplot(x='Married',hue='Gender',data=dataset)

**Insight**
* Married people have taken more loan as compare to unmarried people.
* Less Married Female have taken less loan than Unmarried Female.
* Married Male have taken more loan than female irrespective of there Marriage status.

In [ ]:
sns.jointplot(x='LoanAmount',y='ApplicantIncome',kind='reg',data=dataset)

In [ ]:
sns.jointplot(x='Gender',y='ApplicantIncome',kind='scatter',data=dataset)

In [ ]:
sns.jointplot(x='Gender',y='Property_Area',kind='scatter',data=dataset)

In [ ]:
sns.jointplot(x='Gender',y='LoanAmount',kind='scatter',data=dataset)

In [ ]:
sns.displot(dataset['Property_Area'])

**Insight**
* People have higher number of property in SemiUrban then Urban then Rural area.
* Men have taken larger number of loan than Female.
* Large number of people have taken loan from 100-200

## X ,Y Split

In [ ]:
X = dataset.drop(columns=['Loan_ID','Loan_Status'],axis=1)
Y = dataset['Loan_Status']

In [ ]:
print(X)

In [ ]:
print(Y)

## Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X)

In [ ]:
standardized_data = scaler.transform(X)
standardized_data

In [ ]:
X = standardized_data
Y = dataset['Loan_Status']

In [ ]:
X

In [ ]:
Y

## Performing train and test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,stratify=Y,random_state=2)

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

## Model Selection

In [ ]:
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [ ]:
!pip install xgboost

In [ ]:
models = {
    "svc":SVC(kernel='linear'),
    "Logistic":LogisticRegression(),
    "xgb":xgb.XGBClassifier(),
    "DecisionTree":DecisionTreeClassifier(),
    "RandomForest":RandomForestClassifier(n_estimators=100),
    "NaiveBayes":GaussianNB(),
    "KNN":KNeighborsClassifier(n_neighbors=5),
    "GBC":GradientBoostingClassifier(),

}

In [ ]:
params = {
    "svc":{},
    "Logistic":{},
    "xgb":{},
    "DecisionTree":{},
    "RandomForest":{},
    "NaiveBayes":{},
    "KNN":{},
    "GBC":{}
}

In [ ]:
def evaluate_models(X_train, Y_train,X_test,Y_test,models,param):
        report = {}

        for i in range(len(list(models))):
            model = list(models.values())[i]
            para=param[list(models.keys())[i]]

            gs = GridSearchCV(model,para,cv=3)
            gs.fit(X_train,Y_train)

            # logging.info(f'Hyperparameter {gs} filled')

            model.set_params(**gs.best_params_)
            model.fit(X_train,Y_train)


            #model.fit(X_train, y_train)  # Train model

            y_train_pred = model.predict(X_train)

            y_test_pred = model.predict(X_test)

            train_model_score = accuracy_score(Y_train, y_train_pred)

            test_model_score = accuracy_score(Y_test, y_test_pred)

            report[list(models.keys())[i]] = test_model_score

        return report

In [ ]:
model_report:dict=evaluate_models(X_train,Y_train,X_test,Y_test,models,params)

## To get best model score from dict
best_model_score = max(sorted(model_report.values()))

## To get best model name from dict

best_model_name = list(model_report.keys())[
list(model_report.values()).index(best_model_score)
]
best_model = models[best_model_name]

print("This is the best model:")
print(best_model_name)

model_names = list(params.keys())

actual_model=""

for model in model_names:
    if best_model_name == model:
        actual_model = actual_model + model

        best_params = params[actual_model]

<!-- MODEL PREDICT -->

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np  # for handling NaN

# 👤 Name Input
name_input = widgets.Text(description='Your Name:')

# 👇 Dropdowns and Inputs
gender = widgets.Dropdown(options=['Male', 'Female'], description='Gender:')
married = widgets.Dropdown(options=['Yes', 'No'], description='Married:')
dependents = widgets.Dropdown(options=['0', '1', '2', '3+'], description='Dependents:')
education = widgets.Dropdown(options=['Graduate', 'Not Graduate'], description='Education:')
self_employed = widgets.Dropdown(options=['Yes', 'No'], description='Self Employed:')
applicant_income = widgets.FloatText(description='Income:')
coapplicant_income = widgets.FloatText(description='Co-income:')
loan_amount = widgets.FloatText(description='Loan Amt:')
loan_term = widgets.IntText(description='Term (days):', value=360)

# ✅ Updated Credit History with Yes/No/None
credit_history = widgets.Dropdown(
    options=[
        ('Yes (Good Credit History)', 1.0),
        ('No (Bad Credit History)', 0.0),
        ('None (No Previous Loan)', -1.0) # Changed np.nan to -1.0
    ],
    description='Credit Hist:'
)

property_area = widgets.Dropdown(options=['Urban', 'Semiurban', 'Rural'], description='Area:')

# 🔘 Predict Button & Output
predict_btn = widgets.Button(description="Predict Loan Status", button_style='success')
output = widgets.Output()

# 🧠 Predict Function
def on_predict_clicked(b):
    with output:
        clear_output()
        user_name = name_input.value.strip()

        # Default name if empty
        if user_name == "":
            user_name = "Applicant"

        # Handle NaN (None) credit history by assuming average (e.g., 1.0)
        credit_val = credit_history.value
        if credit_val == -1.0: # Check for -1.0 instead of np.nan
            credit_val = 1.0  # 👈 Tum chaaho to 0.0 bhi set kar sakte ho

        # Map values
        input_dict = {
            'Gender': 1 if gender.value == 'Male' else 0,
            'Married': 1 if married.value == 'Yes' else 0,
            'Dependents': {'0': 0, '1': 1, '2': 2, '3+': 3}[dependents.value],
            'Education': 0 if education.value == 'Graduate' else 1,
            'Self_Employed': 1 if self_employed.value == 'Yes' else 0,
            'ApplicantIncome': applicant_income.value,
            'CoapplicantIncome': coapplicant_income.value,
            'LoanAmount': loan_amount.value,
            'Loan_Amount_Term': loan_term.value,
            'Credit_History': credit_val,
            'Property_Area': {'Urban': 2, 'Semiurban': 1, 'Rural': 0}[property_area.value]
        }

        # Predict
        input_df = pd.DataFrame([input_dict])
        pred = model.predict(input_df)[0]

        # Show personalized result
        if pred == 1:
            print(f"✅ {user_name}, your loan is Approved!")
        else:
            print(f"❌ {user_name}, your loan is Rejected.")

# Button click event
predict_btn.on_click(on_predict_clicked)
# 🔽 Display all widgets
display(
    name_input, gender, married, dependents, education, self_employed,
    applicant_income, coapplicant_income, loan_amount,
    loan_term, credit_history, property_area,
    predict_btn, output
)
